# K4 — route-based interface pick

**Checks:** with no manual interface, the sniffer binds the adapter the OS routes through and says so
(`Auto-Targeting Network Interface: <ip> (default route | first physical adapter)`). The notebook starts the app with the sniffer
on and **reads that line from the bridge** -- nothing to paste -- then compares it with `route print` and the adapter list.

It looks at **this machine** to see which situation it is in and runs the check that fits (the others are `skip`):
1. **Two live adapters** (Ethernet + Wi-Fi) → `(default route)`, a physical adapter.
2. **Full-tunnel VPN on** → the *physical* adapter, `(first physical adapter)`. **The open question:** if chat is only seen on the
   VPN adapter, the "virtual adapters are not trusted" rule is wrong.
3. **Offline** → no crash, the system tab says there is no interface.

Turning the VPN on or the network off is physical: do it, then run the notebook again. "Chat is captured" needs game traffic
(rows `skip`). Needs Node 20+ for the bridge; run Jupyter as Administrator; no *manual interface* set in the app's settings.
The same pipeline without Jupyter: `python -m runbook.run interface`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import bridge, common
from runbook.common import Recorder
from runbook.pipelines.interface import InterfacePick
rec = Recorder("interface")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
bridge.ensure_installed()                # npm ci in runbook/bridge, the first time

In [ ]:
EXE = common.ask_exe()          # target/release/resonance-stream.exe of this checkout, unless RUNBOOK_TEXT_LOCAL_EXE says otherwise
InterfacePick(rec, EXE, common.RUNS).run()

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())